Install Library

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

Unduh data dan jalankan portal lokal

In [2]:
# H-2 (Sel 1) — Unduh CSV judul berita detik.com
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


In [3]:
# H-2 (Sel 2) — Portal berita lokal (siap pakai)
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


In [5]:
# H-2 (Sel 2) — Portal berita lokal (siap pakai)
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):          # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

try:
    server.shutdown(); server.server_close()
except NameError:
    pass

for PORT in range(8000, 8050):
    try:
        server, N_HALAMAN = jalankan_portal(df_sumber, port=PORT)
        break
    except OSError:
        continue

BASE = f"http://127.0.0.1:{PORT}"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


Crawling portal lokal

In [6]:
# H-3 (Sel 1) — Baca robots.txt
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [7]:
# H-3 (Sel 2) — parse_halaman + crawl_iter
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


Crawling sitemap detik.com

In [8]:
# H-4 Langkah 1 — parse_sitemap_news
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None,          # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

In [10]:
# H-4 Langkah 2 — Uji parser pada contoh XML
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


In [11]:
# H-4 Langkah 3 — Fungsi pengambilan yang sopan
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: ghazifadhlullah31@gmail.com)"
DELAY = 2.0
BATAS_PER_BAGIAN = 7
BATAS_TOTAL = 20
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [12]:
# H-4 Langkah 4
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                                      judul kategori    tanggal
  Polri Jerat 20 Korporasi Tersangka Karhutla, Legislator PDIP: Hukum Berat     news 2026-10-30
          KPK Ungkap Perkara Cukai Seret PT Djarum Rugikan Negara Triliunan     news 2026-10-30
MBG Diusulkan Masuk Kurikulum, Komisi X DPR: Perlu Dikaji Serius Urgensinya     news 2026-10-30


Channel ala Flume ke lapisan_mentah

In [13]:
# H-5 — Source -> channel (queue) -> sink (JSONL) untuk 911 halaman portal lokal
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 49.7 detik untuk 911 halaman


Quality, cleaning, dan Parquet berpartisi

In [14]:
# H-6 (Sel 1) — Aturan kualitas
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"     : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"        : jml_kata.between(3, 30),
        "kategori dikenal"       : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"          : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"      : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat"   : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,}  ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0  ( 0.00%)
judul 3-30 kata          gagal:     78  ( 0.09%)
kategori dikenal         gagal:      0  ( 0.00%)
tanggal valid            gagal:      0  ( 0.00%)
url artikel valid        gagal:  5,640  ( 6.20%)
judul tidak duplikat     gagal:    235  ( 0.26%)


In [15]:
# H-6 (Sel 2) — Pembersihan
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [16]:
# H-6 (Sel 3) — Parquet berpartisi + partition pruning
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


Query DuckDB dan bentuk dokumen

In [17]:
# H-7 — DuckDB pada lapisan terkurasi + bentuk dokumen
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


Mini MapReduce: kasus kantin

In [18]:
# H-8 — Map, shuffle, reduce, combiner pada data kantin
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


Word count terdistribusi

In [19]:
# H-9 — Partisi Parquet sebagai input split
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


Combiner dan jumlah data yang berpindah

In [20]:
# H-10 — 1 vs 2 proses, tanpa vs dengan combiner
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.537              825,601
1 proses, dengan combiner         0.547               74,009
2 proses, tanpa combiner          1.200              825,601
2 proses, dengan combiner         1.022               74,009

Hasil dengan combiner sama persis: True


Data skew dan beban reducer

In [21]:
# H-11 — Partitioner (CRC32) dan beban per reducer

R = 4         # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


12. Hadoop Streaming

In [22]:
# H-12 (Sel 1) — Tulis masukan.tsv, mapper.py, reducer.py
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [23]:
# H-12 (Sel 2) — cat | mapper | sort | reducer
!head -n 3 masukan.tsv | python3 mapper.py

!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m1.890s
user	0m2.015s
sys	0m0.072s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [24]:
# H-12 (Sel 3) — Bukti hasil pipe = hasil H-9
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


In [25]:
%%bash
sudo apt-get install -y openjdk-11-jdk-headless > /dev/null
wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
export HADOOP_HOME=$PWD/hadoop-3.3.6
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar \
  $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop

ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 && echo "SAMA"

part-00000
_SUCCESS
SAMA


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
2026-10-30 01:57:14,768 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-10-30 01:57:15,001 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-10-30 01:57:15,001 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-10-30 01:57:15,031 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-10-30 01:57:15,321 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-30 01:57:15,346 INFO mapreduce.JobSubmitter: number of splits:1

Kata teratas per kategori dan ekspor ala Sqoop

In [26]:
# H-13 — Key (kategori, kata), combiner di mapper, ekspor ke SQLite
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))
for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("\nBaris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


LATIHAN

In [27]:
# LATIHAN 1 — Tambah 5 stopword domain, bandingkan top-5 tiap kategori

TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_DOMAIN = STOPWORD | TAMBAHAN
print("Kata yang ditambahkan:", sorted(TAMBAHAN))

def top5_per_kategori(stopword):
    hasil = {}
    for path in splits:
        kat = path.parent.name.split("=")[1]
        c = collections.Counter()
        for t in pd.read_parquet(path, columns=["judul"])["judul"]:
            c.update(k for k in tokenisasi(t) if k not in stopword)
        hasil[kat] = [k for k, _ in c.most_common(5)]
    return hasil

sebelum = top5_per_kategori(STOPWORD)
sesudah = top5_per_kategori(STOPWORD_DOMAIN)
for kat in sorted(sebelum):
    keluar = [k for k in sebelum[kat] if k not in sesudah[kat]]
    masuk = [k for k in sesudah[kat] if k not in sebelum[kat]]
    print(f"{kat:<8} sebelum: {', '.join(sebelum[kat]):<45} sesudah: {', '.join(sesudah[kat]):<45} keluar={keluar} masuk={masuk}")

Kata yang ditambahkan: ['foto', 'galeri', 'ri', 'rp', 'the']
finance  sebelum: corona, rp, ri, harga, jokowi                 sesudah: corona, harga, jokowi, mau, ekonomi           keluar=['rp', 'ri'] masuk=['mau', 'ekonomi']
food     sebelum: makanan, makan, foto, galeri, enak            sesudah: makanan, makan, enak, bikin, rumah            keluar=['foto', 'galeri'] masuk=['bikin', 'rumah']
health   sebelum: corona, virus, pasien, indonesia, meninggal   sesudah: corona, virus, pasien, indonesia, meninggal   keluar=[] masuk=[]
hot      sebelum: corona, anak, rumah, meninggal, the           sesudah: corona, anak, rumah, meninggal, film          keluar=['the'] masuk=['film']
inet     sebelum: corona, virus, ponsel, bikin, indonesia       sesudah: corona, virus, ponsel, bikin, indonesia       keluar=[] masuk=[]
news     sebelum: corona, polisi, kasus, warga, positif         sesudah: corona, polisi, kasus, warga, positif         keluar=[] masuk=[]
oto      sebelum: mobil, corona, motor, rp

In [28]:
# LATIHAN 2 — Key (bulan, kata): apa yang berubah dari Januari ke Mei?
def mapper_bulan(path):
    d = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(d["judul"], d["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(b, kata)] += 1
    return c

per_bulan = collections.Counter()
for c in map(mapper_bulan, splits):
    per_bulan.update(c)

def top_bulan(bulan, n=10):
    return sorted(((v, k) for (b, k), v in per_bulan.items() if b == bulan), reverse=True)[:n]

jan, mei = top_bulan("2020-01"), top_bulan("2020-05")
print("Januari 2020:", ", ".join(f"{k} ({v:,})" for v, k in jan))
print("Mei 2020    :", ", ".join(f"{k} ({v:,})" for v, k in mei))
kata_jan, kata_mei = {k for _, k in jan}, {k for _, k in mei}
print("\nHilang dari 10 teratas di Mei :", sorted(kata_jan - kata_mei))
print("Baru masuk 10 teratas di Mei  :", sorted(kata_mei - kata_jan))
print("Tetap di 10 teratas           :", sorted(kata_jan & kata_mei))

Januari 2020: banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415), indonesia (410)
Mei 2020    : corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), rp (558), psbb (522), covid (522), polisi (517)

Hilang dari 10 teratas di Mei : ['banjir', 'baru', 'indonesia', 'jokowi', 'kpk', 'minta', 'mobil', 'tahun']
Baru masuk 10 teratas di Mei  : ['corona', 'covid', 'kasus', 'lebaran', 'pandemi', 'positif', 'psbb', 'warga']
Tetap di 10 teratas           : ['polisi', 'rp']


In [29]:
# LATIHAN 3 — Mapper bigram (pasangan kata berurutan)
def mapper_bigram(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    pasangan = []
    for t in judul:
        k = tokenisasi(t)
        pasangan += [(f"{a} {b}", 1) for a, b in zip(k, k[1:])]
    return pasangan

hasil_bigram = [mapper_bigram(p) for p in splits]
bigram = collections.Counter()
for h in hasil_bigram:
    for kunci, n in h:
        bigram[kunci] += n

print("Pasangan bigram dihasilkan mapper:", f"{sum(len(h) for h in hasil_bigram):,}")
print("Bigram unik                      :", f"{len(bigram):,}")
print("\n10 bigram teratas:")
for k, n in bigram.most_common(10):
    print(f"  {k:<20} {n:>6,}")
print("\n10 bigram teratas tanpa stopword di kedua kata:")
for k, n in [(k, n) for k, n in bigram.most_common() if not set(k.split()) & STOPWORD][:10]:
    print(f"  {k:<20} {n:>6,}")

Pasangan bigram dihasilkan mapper: 734,783
Bigram unik                      : 349,114

10 bigram teratas:
  virus corona          2,915
  corona di             2,178
  positif corona        1,804
  di tengah             1,258
  pandemi corona        1,075
  di rumah              1,047
  new normal              917
  di indonesia            871
  rp juta                 700
  hari ini                627

10 bigram teratas tanpa stopword di kedua kata:
  virus corona          2,915
  positif corona        1,804
  pandemi corona        1,075
  new normal              917
  rp juta                 700
  rp m                    606
  pasien corona           600
  cegah corona            583
  rp t                    549
  tengah pandemi          513


In [30]:
# LATIHAN 4 — Beban reducer terberat untuk R = 2, 8, 32 (+4 sebagai acuan)
pasangan_combiner = [list(c.items()) for c in h2]
baris_l4 = []
for r in (2, 4, 8, 32):
    b1 = beban(hasil_map, r)
    b2 = beban(pasangan_combiner, r)
    baris_l4.append({"reducer": r, "bagian adil (%)": round(100 / r, 2),
                     "terberat tanpa combiner (%)": round(max(b1) / sum(b1) * 100, 2),
                     "terberat dengan combiner (%)": round(max(b2) / sum(b2) * 100, 2),
                     "rasio terberat/adil (tanpa combiner)": round(max(b1) / sum(b1) * r, 2)})
print(pd.DataFrame(baris_l4).to_string(index=False))

 reducer  bagian adil (%)  terberat tanpa combiner (%)  terberat dengan combiner (%)  rasio terberat/adil (tanpa combiner)
       2            50.00                        52.27                         50.23                                  1.05
       4            25.00                        29.17                         25.55                                  1.17
       8            12.50                        16.83                         12.85                                  1.35
      32             3.12                         5.82                          3.44                                  1.86


In [31]:
# LATIHAN 5 — Jeda 0,05 detik untuk 50 halaman, lalu estimasi 911 halaman dengan jeda 1 detik
JEDA = 0.05
t0 = time.perf_counter()
hasil_l5 = list(crawl_iter(BASE, range(1, 51), delay=JEDA))
waktu_l5 = time.perf_counter() - t0

per_halaman_tanpa_jeda = (waktu_l5 - 50 * JEDA) / 50
estimasi_911 = N_HALAMAN * (per_halaman_tanpa_jeda + 1.0)
print(f"Judul terambil           : {len(hasil_l5):,}")
print(f"Waktu 50 halaman (0,05 s): {waktu_l5:.2f} detik  (jeda saja = {50*JEDA:.2f} detik)")
print(f"Kerja per halaman        : {per_halaman_tanpa_jeda*1000:.1f} ms")
print(f"Estimasi 911 hal, jeda 1s: {estimasi_911:,.0f} detik = {estimasi_911/60:.1f} menit")

Judul terambil           : 5,000
Waktu 50 halaman (0,05 s): 5.80 detik  (jeda saja = 2.50 detik)
Kerja per halaman        : 66.0 ms
Estimasi 911 hal, jeda 1s: 971 detik = 16.2 menit


In [32]:
# LATIHAN 6 — Ganti satu sitemap dengan sitemap_news bagian lain dari sitemap.xml
sesi = requests.Session(); sesi.headers["User-Agent"] = UA_DETIK
host_lama = {urlparse(u).netloc for u in SITEMAP_DIPILIH}
pengganti = None

for indeks in ["https://www.detik.com/sitemap.xml", "https://news.detik.com/sitemap.xml"]:
    try:
        sup = BeautifulSoup(ambil_xml(indeks, sesi), "xml")
    except Exception as e:
        print("Gagal membaca", indeks, "->", type(e).__name__, e)
        continue
    daftar = [l.get_text(strip=True) for l in sup.find_all("loc")]
    news_lain = [u for u in daftar if u.endswith("sitemap_news.xml") and urlparse(u).netloc not in host_lama]
    print(f"{indeks}: {len(daftar)} entri, {len(news_lain)} sitemap_news dari host lain")
    if news_lain:
        pengganti = news_lain[0]
        break

if pengganti:
    SITEMAP_DIPILIH = SITEMAP_DIPILIH[:2] + [pengganti]
    print("Sitemap ke-3 diganti menjadi:", pengganti)
    try:
        sampel_baru = crawl_detik()
    except Exception as e:
        sampel_baru = []
        print("Crawling gagal:", type(e).__name__, "-", e)
    if sampel_baru:
        cek6 = pd.DataFrame(sampel_baru)
        print("\nKategori yang muncul:", cek6["kategori"].value_counts().to_dict())
        print(f"\nAturan kualitas H-6 pada {len(cek6)} judul hasil crawl:")
        for nama, lulus in aturan_kualitas(cek6).items():
            print(f"  {nama:<24} gagal: {(~lulus).sum():>3}")
else:
    print("Tidak menemukan sitemap pengganti.")

https://www.detik.com/sitemap.xml: 536 entri, 191 sitemap_news dari host lain
Sitemap ke-3 diganti menjadi: https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml

Kategori yang muncul: {'news': 7, 'inet': 7, 'finance': 6}

Aturan kualitas H-6 pada 20 judul hasil crawl:
  judul tidak kosong       gagal:   0
  judul 3-30 kata          gagal:   0
  kategori dikenal         gagal:   0
  tanggal valid            gagal:   0
  url artikel valid        gagal:   0
  judul tidak duplikat     gagal:   0


In [33]:
# LATIHAN 7 — Mapper & reducer kata kunci dari hasil crawl H-4
sumber_l7 = sampel if sampel else parse_sitemap_news(CONTOH_XML)
print("Sumber:", "hasil crawl H-4" if sampel else "CONTOH_XML (crawl H-4 gagal/kosong)", "|", len(sumber_l7), "berita")

def mapper_kunci(berita):
    return [(k.strip().lower(), 1) for k in berita.get("kata_kunci", "").split(",") if k.strip()]

def reducer_kunci(kunci, nilai):
    return kunci, sum(nilai)

# MAP
pasangan_l7 = [p for b in sumber_l7 for p in mapper_kunci(b)]
# SHUFFLE
acak_l7 = collections.defaultdict(list)
for k, n in pasangan_l7:
    acak_l7[k].append(n)
# REDUCE
hasil_l7 = dict(reducer_kunci(k, v) for k, v in acak_l7.items())

print("Berita yang punya kata kunci:", sum(1 for b in sumber_l7 if b.get("kata_kunci")))
print("Pasangan dari mapper        :", len(pasangan_l7))
print("Kata kunci unik             :", len(hasil_l7))
print("\nKata kunci terbanyak:")
for k, n in sorted(hasil_l7.items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(f"  {k:<30} {n}")

Sumber: hasil crawl H-4 | 20 berita
Berita yang punya kata kunci: 20
Pasangan dari mapper        : 92
Kata kunci unik             : 81

Kata kunci terbanyak:
  ibl                            4
  bareskrim polri                2
  ibl all indonesian             2
  ibl all indonesian 2026        2
  karhutla                       2
  kecerdasan buatan              2
  korupsi                        2
  polri                          2
  samsung solve for tomorrow     2
  aerospora                      1


TUGAS MANDIRI: Inverted Index

In [34]:
# TUGAS MANDIRI — Inverted index dengan MapReduce
# Kategori inet tidak punya id (url rusak) -> diberi id pengganti "inet-<crc32 judul>".
# Judul sudah unik setelah drop_duplicates, jadi id pengganti stabil dan tidak bentrok.

def id_dokumen(id_asli, judul, kategori):
    if pd.notna(id_asli):
        return str(int(id_asli))
    return f"{kategori}-{zlib.crc32(judul.encode()):08x}"

def mapper_inverted(path):
    """Satu split -> pasangan (kata, id). Kata yang sama dalam satu judul hanya dikirim sekali."""
    d = pd.read_parquet(path, columns=["judul", "id"])
    kategori = path.parent.name.split("=")[1]
    pasangan = []
    for judul, i in zip(d["judul"], d["id"]):
        doc = id_dokumen(i, judul, kategori)
        pasangan += [(kata, doc) for kata in set(tokenisasi(judul))]
    return pasangan

def reducer_inverted(kata, daftar_id):
    """Gabungkan id menjadi daftar terurut (id angka dulu, lalu id pengganti)."""
    return kata, sorted(set(daftar_id), key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else 0, x))

# MAP
hasil_map_inv = [mapper_inverted(p) for p in splits]
print("Pasangan (kata, id) dari mapper:", f"{sum(len(h) for h in hasil_map_inv):,}")

# SHUFFLE
acak_inv = collections.defaultdict(list)
for h in hasil_map_inv:
    for kata, doc in h:
        acak_inv[kata].append(doc)

# REDUCE
inverted = dict(reducer_inverted(k, v) for k, v in acak_inv.items())
print("Kata di inverted index          :", f"{len(inverted):,}")
print("Judul inet dengan id pengganti  :", (bersih["kategori"] == "inet").sum())

# Tiga kata contoh beserta lima id pertama
for kata in ["corona", "jokowi", "vaksin"]:
    ids = inverted.get(kata, [])
    print(f"\n{kata:<8} muncul di {len(ids):,} judul | 5 id pertama: {ids[:5]}")
    for i in ids[:2]:
        if i.isdigit():
            print("   ", i, "->", bersih.loc[bersih["id"] == int(i), "judul"].iloc[0])

# Contoh kata dari kategori inet (memakai id pengganti)
contoh_inet = [i for i in inverted.get("ponsel", []) if not i.isdigit()][:5]
print("\nponsel (id pengganti inet):", contoh_inet)

# Simpan ke siap_pakai/
inv_df = pd.DataFrame({"kata": list(inverted.keys()),
                       "jumlah_judul": [len(v) for v in inverted.values()],
                       "daftar_id": list(inverted.values())})
inv_df.to_parquet(SIAP / "inverted_index.parquet", index=False)
print("\nTersimpan:", SIAP / "inverted_index.parquet", "|", len(inv_df), "baris")

Pasangan (kata, id) dari mapper: 817,993
Kata di inverted index          : 32,275
Judul inet dengan id pengganti  : 5577

corona   muncul di 17,228 judul | 5 id pertama: ['4856495', '4856598', '4868317', '4868545', '4869163']
    4856495 -> WHO Sebut Penyakit Misterius di China Disebabkan oleh Virus Corona
    4856598 -> Seberapa Berbahaya Virus Corona di Balik 'Penyakit Misterius' di China?

jokowi   muncul di 2,316 judul | 5 id pertama: ['4842674', '4842690', '4842703', '4842774', '4843215']
    4842674 -> Jakarta Banjir, Jokowi: Nomorsatukan Keselamatan Warga
    4842690 -> Halim hingga Tol Cikampek Banjir, Jokowi Instruksikan Normalisasi

vaksin   muncul di 188 judul | 5 id pertama: ['4862824', '4873864', '4874997', '4875084', '4875552']
    4862824 -> Heboh Pneumonia 'Misterius' China, Perlukah Minta Vaksin?
    4873864 -> Vaksin Corona Virus, Baru Akan Diuji Coba pada April 2020

ponsel (id pengganti inet): ['inet-005bfd65', 'inet-00c757d7', 'inet-00db5396', 'inet-017dc349', 'ine

ZIP untuk pengumpulan

In [35]:
import zipfile
from google.colab import files

with zipfile.ZipFile("praktikum5_hasil.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["lapisan_mentah", "lapisan_terkurasi", "siap_pakai", "contoh_crawl"]:
        for p in Path(folder).rglob("*"):
            if p.is_file():
                z.write(p)
print("Ukuran zip:", round(os.path.getsize("praktikum5_hasil.zip") / 1e6, 1), "MB")
files.download("praktikum5_hasil.zip")

Ukuran zip: 15.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

DIR_DRIVE = "/content/drive/MyDrive/BigData/Praktikum5"
os.makedirs(DIR_DRIVE, exist_ok=True)

for folder in ["lapisan_mentah", "lapisan_terkurasi", "siap_pakai", "contoh_crawl"]:
    if os.path.exists(folder):
        tujuan = f"{DIR_DRIVE}/{folder}"
        shutil.rmtree(tujuan, ignore_errors=True)
        shutil.copytree(folder, tujuan)
        print("Disalin:", tujuan)

for f in ["mapper.py", "reducer.py", "hasil_pipe.txt"]:
    if os.path.exists(f):
        shutil.copy(f, DIR_DRIVE)

zip_path = f"{DIR_DRIVE}/praktikum5_hasil.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["lapisan_mentah", "lapisan_terkurasi", "siap_pakai", "contoh_crawl"]:
        for p in Path(folder).rglob("*"):
            if p.is_file():
                z.write(p)
print("\nZip tersimpan:", zip_path, "|", round(os.path.getsize(zip_path) / 1e6, 1), "MB")

print("\nIsi folder Drive Praktikum5:")
for f in sorted(os.listdir(DIR_DRIVE)):
    print(" -", f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Disalin: /content/drive/MyDrive/BigData/Praktikum5/lapisan_mentah
Disalin: /content/drive/MyDrive/BigData/Praktikum5/lapisan_terkurasi
Disalin: /content/drive/MyDrive/BigData/Praktikum5/siap_pakai
Disalin: /content/drive/MyDrive/BigData/Praktikum5/contoh_crawl

Zip tersimpan: /content/drive/MyDrive/BigData/Praktikum5/praktikum5_hasil.zip | 15.9 MB

Isi folder Drive Praktikum5:
 - contoh_crawl
 - hasil_pipe.txt
 - lapisan_mentah
 - lapisan_terkurasi
 - mapper.py
 - praktikum5_hasil.zip
 - reducer.py
 - siap_pakai
